In [1]:
!pip install selenium pyarrow lxml requests beautifulsoup4 pandas
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import glob
import pickle
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
}

YEAR = 2025
SAVE_DIR = rf'C:\\Users\\lcsse\\Desktop\\STAGE_GENT\\scrapping\\data\\pcs{YEAR}'
os.makedirs(SAVE_DIR, exist_ok=True)

dates = []
list_of_result_dfs = []
races = []
data_by_month = {}


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


/Users/arthurdeletang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
def is_page_not_found(response_text):
    return "Page not found" in response_text

def process_results_page(response_text, url):
    soup = BeautifulSoup(response_text, 'lxml')

    # Date
    try:
        date_tag = soup.select_one('div.value')
        date = date_tag.get_text(strip=True)
    except Exception:
        date = None
    dates.append(date)

    # Classification (2.UWT, 1.UWT, etc.)
    try:
        classif = None
        for li in soup.find_all('li'):
            if 'Classification:' in li.get_text():
                classif = li.get_text(strip=True).replace('Classification:', '').strip()
                break
    except Exception:
        classif = None

    # Startlist quality score
    try:
        startlist_quality = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Startlist quality score:' in text:
                match = re.search(r'Startlist quality score:\s*(\d+)', text)
                if match:
                    startlist_quality = int(match.group(1))
                break
    except Exception:
        startlist_quality = None

    # Average temperature
    try:
        avg_temperature = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Avg. temperature:' in text:
                match = re.search(r'Avg\. temperature:\s*([\d.]+)', text)
                if match:
                    avg_temperature = float(match.group(1))
                break
    except Exception:
        avg_temperature = None

    # Won how
    try:
        won_how = None
        for li in soup.find_all('li'):
            text = li.get_text()
            if 'Won how:' in text:
                won_how = text.split('Won how:')[-1].strip()
                break
    except Exception:
        won_how = None

    tables = soup.find_all('table')
    if tables:
        dfs = pd.read_html(str(tables))
        if dfs:
            df = dfs[0]
            df['date'] = date
            df['classification'] = classif
            df['startlist_quality'] = startlist_quality
            df['avg_temperature'] = avg_temperature
            df['won_how'] = won_how
            list_of_result_dfs.append(df)
            races.append(url)

def scrape_base_url(base_url):
    def has_results_table(response_text):
        soup = BeautifulSoup(response_text, 'lxml')
        return len(soup.find_all('table')) > 0

    result_url = base_url + "/result"
    response = requests.get(result_url, headers=headers)
    time.sleep(1)
    if has_results_table(response.text):
        process_results_page(response.text, result_url)

    stage = 1
    while True:
        stage_url = f"{base_url}/stage-{stage}"
        response = requests.get(stage_url, headers=headers)
        time.sleep(1)
        if has_results_table(response.text):
            process_results_page(response.text, stage_url)
            stage += 1
        else:
            break

    print(f"✅ {base_url}")

def scrape_urls(url_list):
    for base_url in url_list:
        scrape_base_url(base_url)

def clean_race_url(url):
    url = url.replace('/result', '')
    url = re.sub(r'/stage-\d+', '', url)
    return url

def save_month(month):
    for race, df, date in zip(data_by_month[month]['races'], data_by_month[month]['dfs'], data_by_month[month]['dates']):
        try:
            df['race_url'] = race
            df['date'] = date

            if '/stage-' in race:
                race_type = 'stage_' + race.split('/stage-')[-1]
            elif '/result' in race:
                race_type = 'result'
            else:
                race_type = 'one_day'

            race_name = race.replace('https://www.procyclingstats.com/race/', '')
            race_name = re.sub(r'/stage-\d+', '', race_name)
            race_name = race_name.replace('/result', '').replace('/', '_')

            clean_date = date.replace(' ', '_') if date else 'no_date'

            if 'classification' in df.columns and df['classification'].iloc[0]:
                classif = str(df['classification'].iloc[0]).strip()
            else:
                classif = 'no_classif'

            filename = f"{clean_date}__{race_type}__{race_name}__{classif}.csv"
            df.to_csv(os.path.join(SAVE_DIR, filename), index=False)
        except Exception as e:
            print(f"❌ {race}: {e}")
    print(f"✅ Mois {month}: {len(data_by_month[month]['races'])} fichiers sauvegardés")

In [3]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=1&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour janvier 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[1] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(1)
driver.quit()

✅ 19 URLs trouvées pour janvier 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-australia-itt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/vuelta-al-tachira/2025


...

✅ https://www.procyclingstats.com/race/race-torquay/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/deia-trophy/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-ecuador-itt/2025
Total brut: 43 | Uniques: 19 | Manquantes: 0
✅ Mois 1: 43 fichiers sauvegardés


In [4]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=2&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour fevrier 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[2] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(2)
driver.quit()

✅ 45 URLs trouvées pour fevrier 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-pollenca-port-d-andratx/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-antalya/2025


...

✅ https://www.procyclingstats.com/race/nc-philippines-itt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/gran-camino/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-philippines/2025
Total brut: 102 | Uniques: 45 | Manquantes: 0
❌ https://www.procyclingstats.com/race/colombia-21/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/colombia-21/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/colombia-21/2025/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/colombia-21/2025/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/colombia-21/2025/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/colombia-21/2025/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-internacional-bio-bio/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-ciclista-internacional-bio-bio/2025/stage-2: single positional inde

In [5]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=3&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mars 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[3] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(3)
driver.quit()

✅ 50 URLs trouvées pour mars 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/omloop-het-nieuwsblad/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/faun-ardeche-classic/2025


...

✅ https://www.procyclingstats.com/race/gp-adria-mobil/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-emilia/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/la-roue-tourangelle/2025
Total brut: 96 | Uniques: 50 | Manquantes: 0
❌ https://www.procyclingstats.com/race/ster-van-zwolle/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/elfsteden-race/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/elfsteden-race/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/classic-loire-atlantique/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/strade-bianche-di-romagna/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-goriska-vipava-valley/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-emilia/2025/result: single positional indexer is out-of-bounds
✅ Mois 3: 96 fichiers sauvegardés


In [6]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=4&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour avril 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[4] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(4)
driver.quit()

✅ 52 URLs trouvées pour avril 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/dwars-door-vlaanderen/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/paris-camembert/2025


...

✅ https://www.procyclingstats.com/race/tour-of-bostonliq/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-de-romandie/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/vuelta-bantrab/2025
Total brut: 126 | Uniques: 52 | Manquantes: 0
✅ Mois 4: 126 fichiers sauvegardés


In [7]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=5&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour mai 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[5] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(5)
driver.quit()

✅ 56 URLs trouvées pour mai 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/eschborn-frankfurt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/omloop-van-het-waasland/2025


...

✅ https://www.procyclingstats.com/race/tour-of-estonia/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-malaysia-itt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grote-prijs-marcel-kint/2025
Total brut: 123 | Uniques: 56 | Manquantes: 0
❌ https://www.procyclingstats.com/race/ronde-van-overijssel/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/beskid-classic/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2025/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/joe-martin-stage-race-me/2025/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/veenendaal-veenendaal/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/circuit-de-wallonie/

In [8]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=6&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juin 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[6] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(6)
driver.quit()

✅ 189 URLs trouvées pour juin 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/klaipeda-grand-prix/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-malaysia/2025


...

✅ https://www.procyclingstats.com/race/national-championships-malta-me-road-race/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/national-championships-iceland---road-race/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-indonesia/2025
Total brut: 242 | Uniques: 188 | Manquantes: 1
❌ https://www.procyclingstats.com/race/klaipeda-grand-prix/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-cameroun/2025/stage-7: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2025/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/ronde-de-l-oise/2025/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/zlm-tour/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/zlm-tour/2025/stage-2: single positional indexer is out-of-bounds
❌ https://

In [9]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=7&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour juillet 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[7] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(7)
driver.quit()

✅ 44 URLs trouvées pour juillet 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/trofeo-citta-di-brescia/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-sweden-itt/2025
✅ https://www.procyclingstats.com/race/course-cycliste-de-solidarnosc/2025


...

✅ https://www.procyclingstats.com/race/grand-prix-de-la-ville-de-perenchies/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/banyuwangi-tour-de-ijen/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-alsace/2025
Total brut: 94 | Uniques: 43 | Manquantes: 1
❌ https://www.procyclingstats.com/race/gp-de-la-ville-de-nogent-sur-oise/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/sd-worx-bw-classic/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/nc-kyrgyzstan/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/baltyk-karkonosze-tour/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/baltyk-karkonosze-tour/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/gp-czech-republic/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/national-championships-sint-maarten-road-race/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/classique-des-ca

In [11]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=8&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour aout 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[8] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(8)
driver.quit()

✅ 44 URLs trouvées pour aout 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/vuelta-pilsen-a-colombia/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-cycliste-international-de-la-guadeloupe/2025


...

✅ https://www.procyclingstats.com/race/grand-prix-de-plouay/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/grote-prijs-stad-halle/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/gp-de-la-somme/2025
Total brut: 170 | Uniques: 43 | Manquantes: 1
❌ https://www.procyclingstats.com/race/vuelta-pilsen-a-colombia/2025/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-pilsen-a-colombia/2025/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/cupa-max-ausnit/2025/result: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-konya/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-konya/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-konya/2025/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-konya/2025/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/memorial-henryka-lasaka/2025/result: single positional indexer 

In [12]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=9&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour septembre 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[9] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(9)
driver.quit()

✅ 47 URLs trouvées pour septembre 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-de-maurice/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-of-britain/2025


...

✅ https://www.procyclingstats.com/race/world-championship/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-de-langkawi/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecate

✅ https://www.procyclingstats.com/race/tour-of-croatia/2025
Total brut: 127 | Uniques: 47 | Manquantes: 0
❌ https://www.procyclingstats.com/race/tour-de-maurice/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-maurice/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-maurice/2025/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-de-maurice/2025/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/giro-della-regione-friuli-venezia-giulia/2025/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-maroc/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-maroc/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-du-maroc/2025/stage-3: single positional inde

In [13]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=10&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour octobre 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[10] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(10)
driver.quit()

✅ 39 URLs trouvées pour octobre 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/uec-road-european-championships-itt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/munsterland-giro/2025


...

✅ https://www.procyclingstats.com/race/vuelta-a-guatemala/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-israel-itt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-mexico/2025
Total brut: 76 | Uniques: 38 | Manquantes: 1
❌ https://www.procyclingstats.com/race/vuelta-a-ecuador/2025/stage-1: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-ecuador/2025/stage-2: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-ecuador/2025/stage-3: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-ecuador/2025/stage-4: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-ecuador/2025/stage-5: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-ecuador/2025/stage-6: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/vuelta-a-ecuador/2025/stage-7: single positional indexer is out-of-bounds
❌ https://www.procyclingstats.com/race/tour-of-ankara/2025/stage-1: single positional indexer is out-of-bounds

In [14]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=11&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour novembre 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[11] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(11)
driver.quit()

✅ 6 URLs trouvées pour novembre 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/tour-de-okinawa/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/african-continental-championships-itt/2025


...

✅ https://www.procyclingstats.com/race/african-championships/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-taiwan/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/juegos-bolivarianos-me-itt/2025
Total brut: 6 | Uniques: 6 | Manquantes: 0
✅ Mois 11: 6 fichiers sauvegardés


In [15]:
edge_options = Options()
edge_options.add_argument("--disable-gpu")
edge_options.add_argument("--window-size=1920,1080")
driver = webdriver.Edge(options=edge_options)
driver.get('https://www.procyclingstats.com/races.php?season=2025&month=12&category=1&racelevel=&pracelevel=smallerorequal&racenation=&class=&filter=Filter&p=uci&s=calendar-plus-filters')
time.sleep(5)
elements = driver.find_elements(By.CSS_SELECTOR, "table tbody tr td:nth-child(2) a")
urls = [el.get_attribute("href") for el in elements]
print(f"✅ {len(urls)} URLs trouvées pour decembre 2025")
scrape_urls(urls)
cleaned_races = [clean_race_url(r) for r in races]
unique_races = set(cleaned_races)
print(f"Total brut: {len(races)} | Uniques: {len(unique_races)} | Manquantes: {len(set(urls) - unique_races)}")
data_by_month[12] = {"races": races[:], "dfs": list_of_result_dfs[:], "dates": dates[:]}
races = []; list_of_result_dfs = []; dates = []
save_month(12)
driver.quit()

✅ 7 URLs trouvées pour decembre 2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/nc-india-itt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/juegos-bolivarianos-me-road-race/2025


...

✅ https://www.procyclingstats.com/race/south-east-asian-games-ttt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/south-east-asian-games-itt/2025


/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_23089/856822101.py:64: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  dfs = pd.read_html(str(tables))


✅ https://www.procyclingstats.com/race/south-east-asian-games/2025
Total brut: 16 | Uniques: 7 | Manquantes: 0
✅ Mois 12: 16 fichiers sauvegardés
